# TF-IDF Vectorization and Similarity Analysis

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.columns.tolist())

Rows: 4005
Columns: 23
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'topics_clean', 'description_clean', 'combined_text']


In [3]:
print("Has title:", "title" in df.columns)
print("Has combined_text:", "combined_text" in df.columns)

empty_text = df["combined_text"].fillna("").str.strip() == ""
print("Empty combined_text rows:", empty_text.sum())
print("Usable records:", (~empty_text).sum())

Has title: True
Has combined_text: True
Empty combined_text rows: 0
Usable records: 4005


In [4]:
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["combined_text"].fillna(""))
print("Matrix shape:", tfidf_matrix.shape)

Matrix shape: (4005, 71129)


In [5]:
words = vectorizer.get_feature_names_out()
print("Total vocabulary size:", len(words))
print("First 20 words:", words[:20])
print("Random 20 words:", np.random.RandomState(42).choice(words, 20, replace=False))

Total vocabulary size: 71129
First 20 words: ['00' '000' '0000' '000000004' '0000001' '000001' '00001' '000042' '0001'
 '00046' '0009' '000th' '001' '0015' '002' '0025' '003' '004' '004867'
 '005']
Random 20 words: ['glacial' 'radford' 'rockyou' 'entrancing' 'moping' 'nagging'
 'preposterously' 'deresiewicz' 'unethical' 'lino' 'maki' 'cytotoxic'
 'discrete' 'kremer' 'terminalia' 'chalta' 'robespierre' 'rezero' 'parody'
 'dampened']


In [6]:
row = tfidf_matrix[0].toarray().flatten()
top_idx = row.argsort()[::-1][:10]
print("Talk:", df.loc[0, "title"])
for i in top_idx:
    print(words[i], round(row[i], 3))

Talk: Averting the climate crisis
tipper 0.29
carbon 0.201
nashville 0.166
slideshow 0.163
shoney 0.158
climate 0.154
emissions 0.135
al 0.132
warming 0.131
restaurant 0.131


In [7]:
scores = cosine_similarity(tfidf_matrix[0], tfidf_matrix).flatten()
print("Number of scores:", len(scores))
print("Score of talk 0 with itself:", round(scores[0], 3))
print("Lowest score:", round(scores.min(), 3))
print("Highest score:", round(scores.max(), 3))

Number of scores: 4005
Score of talk 0 with itself: 1.0
Lowest score: 0.003
Highest score: 1.0


In [8]:
top_idx = scores.argsort()[::-1][1:6]
print("Selected talk:", df.loc[0, "title"])
print()
for i in top_idx:
    print(round(scores[i], 3), "-", df.loc[i, "title"])

Selected talk: Averting the climate crisis

0.266 - New thinking on the climate crisis
0.244 - How we could change the planet's climate future
0.238 - A climate solution where all sides can win
0.235 - Climate change is happening. Here's how we adapt
0.231 - Can we stop climate change by removing CO2 from the air?


In [9]:
def find_talks(keyword, n=8):
    mask = df["title"].str.contains(keyword, case=False, na=False)
    print("Keyword:", keyword)
    print(df.loc[mask, ["title"]].head(n))
    print()

find_talks("artificial intelligence")
find_talks("cancer")
find_talks("school")

Keyword: artificial intelligence
Empty DataFrame
Columns: [title]
Index: []

Keyword: cancer
                                        title
30         Meet the future of cancer research
589       A new strategy in the war on cancer
655              Can we eat to starve cancer?
679                Fighting cancer with dance
836   Understanding cancer through proteomics
975              Fighting a contagious cancer
999               Open-source cancer research
1079     Treating cancer with electric fields

Keyword: school
                                                 title
4                          Do schools kill creativity?
7    Let's teach religion -- all religion -- in sch...
199                        My wish: Once Upon a School
286                   What's wrong with school lunches
662                Inside a school for suicide bombers
775                              My green school dream
841          How state budgets are breaking US schools
978                 A short intro to

In [10]:
def recommend(idx, top_n=5):
    scores = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    top_idx = scores.argsort()[::-1][1:top_n + 1]
    print("Selected talk:", df.loc[idx, "title"])
    for i in top_idx:
        print(round(scores[i], 3), "-", df.loc[i, "title"])
    print()

In [11]:
for talk_number in [0, 589, 4]:
    recommend(talk_number)

Selected talk: Averting the climate crisis
0.266 - New thinking on the climate crisis
0.244 - How we could change the planet's climate future
0.238 - A climate solution where all sides can win
0.235 - Climate change is happening. Here's how we adapt
0.231 - Can we stop climate change by removing CO2 from the air?

Selected talk: A new strategy in the war on cancer
0.606 - A simple new blood test that can catch cancer early
0.554 - Understanding cancer through proteomics
0.525 - Open-source cancer research
0.516 - New nanotech to detect cancer early
0.475 - The future of early cancer detection?

Selected talk: Do schools kill creativity?
0.322 - How to escape education's death valley
0.296 - Bring on the learning revolution!
0.258 - How I stopped the Taliban from shutting down my school
0.257 - Our failing schools. Enough is enough!
0.249 - How to run a company with (almost) no rules



## Activity 5: Observations and Limitations

1. **Scores are not percentages.** The cancer talk's top 5 scored between 0.475 and 0.606, but the climate talk's top 5 scored only 0.231 to 0.266, even though all of them were relevant. So a score can only be used to rank talks for one selected talk. It should not be compared across different talks.

2. **Rare words and names can mislead the scores.** For the climate talk, the highest TF-IDF words included names such as "tipper", "nashville" and "shoney". These are rare, so TF-IDF rewards them, but they say little about the topic. This pulled the similarity scores down.

3. **The method matches words, not meaning.** For the schools talk, "How to run a company with (almost) no rules" was recommended at 0.249. It is not about schools, so the match is probably just shared vocabulary. Similarly, "How I stopped the Taliban from shutting down my school" appears because of the word "school", even though the context is very different.

4. **Numbers became vocabulary.** The vocabulary of 71,129 words includes entries like "00", "000" and "0001". These add noise and make the matrix larger without adding meaning. A later cleaning step could remove them.

5. **A high score does not guarantee the user will like the recommendation.** Similarity only measures how alike the text is. It does not know the viewer's taste, the speaker's style or how interesting the talk is.

## TF-IDF and Cosine Similarity in My Own Words

**TF-IDF** gives every word in a talk a score. A word gets a higher score if it appears often in that talk (TF), and a lower score if it appears in almost every talk (IDF). So common words like "people" score low, and special words like "carbon" or "proteomics" score high. Each talk becomes a row of numbers, one number per word in the vocabulary.

**Cosine similarity** compares two of these rows of numbers and checks how closely they point in the same direction. A score near 1 means the talks use very similar important words. A score near 0 means they share almost nothing.

To recommend talks, I compare the selected talk with all the others, sort the scores from highest to lowest, skip the talk itself (it always scores 1.0 with itself), and take the top 5.